# Smart Compression: Give Sensitive Layers More Bits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codepawl/turboquant-torch/blob/main/examples/03_adaptive_compression.ipynb)

Not all layers are equal. Some are more sensitive to quantization. This notebook runs calibration to find per-layer sensitivity, then compares allocation strategies.

## Install

In [ ]:
!pip install -q "turboquant-torch[hf]" matplotlib

## Load Model & Run Calibration

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from turboquant import TurboQuantKVCache
from turboquant.adaptive import (
    AdaptiveKVCache, calibration_allocation,
    gradient_allocation, uniform_allocation,
)
from turboquant.compat import detect_model_kv_info, extract_kv

MODEL = "HuggingFaceTB/SmolLM2-135M"
CALIBRATION_TEXT = (
    "The transformer architecture uses multi-head attention to process "
    "sequences in parallel. Each attention head computes query, key, and "
    "value projections, then uses scaled dot-product attention to weight "
    "the values. The KV cache stores previously computed keys and values "
    "to avoid redundant computation during autoregressive generation. "
) * 4

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.eval()
info = detect_model_kv_info(model)
print(f"{info.n_layers} layers, head_dim={info.head_dim}, kv_heads={info.num_kv_heads}")

## Per-Layer Sensitivity

In [ ]:
print("Running calibration (forward pass on sample text)...")
calibrated_bits = calibration_allocation(
    model, tokenizer, CALIBRATION_TEXT, target_avg_bits=3.0,
)
avg_bits = sum(calibrated_bits) / len(calibrated_bits)
bit_counts = {}
for b in sorted(set(calibrated_bits)):
    bit_counts[b] = calibrated_bits.count(b)

print(f"Average bits: {avg_bits:.2f}")
print(f"Distribution: {bit_counts}")
print()

for i, b in enumerate(calibrated_bits):
    bar = "X" * b + "." * (4 - b)
    print(f"  Layer {i:2d}: [{bar}] {b}-bit")

## Compare Strategies: Uniform vs Gradient vs Calibrated

In [ ]:
uniform_bits = uniform_allocation(info.n_layers, bit_width=3)
gradient_bits = gradient_allocation(info.n_layers, min_bits=2, max_bits=4, strategy="linear")

strategies = {
    "Uniform 3-bit": uniform_bits,
    "Gradient 2→4 bit": gradient_bits,
    "Calibrated": calibrated_bits,
}

# Get real KV cache
inputs = tokenizer(CALIBRATION_TEXT, return_tensors="pt")
with torch.no_grad():
    out = model(**inputs, use_cache=True)
kv_pairs = extract_kv(out.past_key_values)

# Measure per-layer MSE
strategy_mses = {}
for name, bits in strategies.items():
    layer_mses = []
    for i, (k, v) in enumerate(kv_pairs):
        if k is None:
            layer_mses.append(0.0)
            continue
        cache = TurboQuantKVCache(head_dim=info.head_dim, bit_width=bits[i], residual_length=0)
        compressed = cache.compress(k.float(), v.float())
        k_hat = cache.decompress_keys(compressed)
        mse = ((k.float() - k_hat) ** 2).mean().item()
        layer_mses.append(mse)
    strategy_mses[name] = layer_mses

print(f"{'Strategy':>20}  {'Avg Bits':>9}  {'Total MSE':>10}  {'Max MSE':>10}")
print(f"{'---':>20}  {'---':>9}  {'---':>10}  {'---':>10}")
for name, bits in strategies.items():
    mses = strategy_mses[name]
    avg_b = sum(bits) / len(bits)
    print(f"{name:>20}  {avg_b:>9.2f}  {sum(mses):>10.4f}  {max(mses):>10.4f}")

## Plot: Bit Allocation & MSE per Layer

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
layers = list(range(info.n_layers))

for name, bits in strategies.items():
    ax1.plot(layers, bits, "o-", label=name, markersize=3)
ax1.set_xlabel("Layer")
ax1.set_ylabel("Bit Width")
ax1.set_title("Bit Allocation per Layer")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yticks([2, 3, 4])

for name, mses in strategy_mses.items():
    ax2.plot(layers, mses, "o-", label=name, markersize=3)
ax2.set_xlabel("Layer")
ax2.set_ylabel("Key MSE")
ax2.set_title("Reconstruction Error per Layer")
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle(f"Adaptive Compression — {MODEL}", fontsize=14)
fig.tight_layout()
plt.show()

## Practical Usage with `AdaptiveKVCache.from_model()`

In [ ]:
adaptive = AdaptiveKVCache.from_model(
    model, tokenizer, head_dim=64,  # SmolLM2-135M head_dim
    target_avg_bits=3.0,
)
print(adaptive.summary())